(remote-steps)=
# Remote steps
Use remote steps to access HTTP sources, external URIs, and remote functions. 
This icon in the UI indicates remote steps: <img src="../_static/images/steps-remote.png" alt="graph-steps-remote" width="30"/>.

**In this section**
- [SendToHttp](#sendtohttp)
- [RemoteStep](#remotestep)
- [RemoteFunctionStep](#remotefunctionstep)

## SendToHttp

Joins each event with data from any HTTP source. Used for event augmentation. See {py:class}`~storey.transformations.SendToHttp`.

## RemoteStep
- Description: Use RemoteStep in both sync and async engines to invoke an external URI (HTTP or MLRun function). See {py:class}`~mlrun.serving.remote.RemoteStep`. 
- Examples: Using the `async` engine to trigger an external heavy process, such as a service generating a test using model and storing it in a DB would look similar to:

In [ ]:
%%writefile graph_handlers.py
def passthrough(event):
    return event

In [15]:
from mlrun.serving.remote import RemoteStep

fn_remote = project.set_function(
    name="remote-demo", func="graph_handlers.py", kind="serving", image="mlrun/mlrun"
)

graph_remote = fn_remote.set_topology("flow", engine="async")

# No-op entry step — logs the event before it is forwarded to the remote endpoint
graph_remote.add_step(name="start", handler="passthrough")

# RemoteStep sends the event body as an HTTP request to the configured URL.
# The full HTTP response body replaces the event and is passed to the next step.
graph_remote.add_step(
    RemoteStep(
        name="call_remote",
        url="https://httpbin.org/json",  # GET request to httpbin's demo JSON endpoint
        method="GET",
    ),
    after="start",
).respond()

server_remote = fn_remote.to_mock_server()

for event in [{"value": 8}, {"value": 15}, {"value": 21}]:
    result = server_remote.test("/", event)
    print(result)

{'slideshow': {'author': 'Yours Truly', 'date': 'date of publication', 'slides': [{'title': 'Wake up to WonderWidgets!', 'type': 'all'}, {'items': ['Why <em>WonderWidgets</em> are great', 'Who <em>buys</em> WonderWidgets'], 'title': 'Overview', 'type': 'all'}], 'title': 'Sample Slide Show'}}
{'slideshow': {'author': 'Yours Truly', 'date': 'date of publication', 'slides': [{'title': 'Wake up to WonderWidgets!', 'type': 'all'}, {'items': ['Why <em>WonderWidgets</em> are great', 'Who <em>buys</em> WonderWidgets'], 'title': 'Overview', 'type': 'all'}], 'title': 'Sample Slide Show'}}
{'slideshow': {'author': 'Yours Truly', 'date': 'date of publication', 'slides': [{'title': 'Wake up to WonderWidgets!', 'type': 'all'}, {'items': ['Why <em>WonderWidgets</em> are great', 'Who <em>buys</em> WonderWidgets'], 'title': 'Overview', 'type': 'all'}], 'title': 'Sample Slide Show'}}


# RemoteFunctionStep
- Description: Calls a remote functions. See {py:class}`~mlrun.serving.remote.RemoteFunctionStep`.
- Use Case: Use this step when you want to invoke an **existing function deployed in MLRun** as part of a serving graph without manually specifying its HTTP endpoint.<br>
The step accepts a function name or URI, retrieves the function object from MLRun, and automatically resolves the function’s invocation URL.<br> This simplifies integration between serving graphs and previously deployed functions, especially when the endpoint address may change between environments.<br>
When the step executes, the incoming event is forwarded to the remote function via its resolved HTTP endpoint. The remote function response is forwarded to the next step in the graph.
   ``` {admonition} Note
   - The remote function can belong to a different project.
   - The function must expose an **HTTP trigger**.
   ```
- Example


In [ ]:
# Reference an existing Nuclio function
step = RemoteFunctionStep(fn="my-nuclio-function", project_name="my-project")

# Create a serving function
serving_fn = mlrun.new_function(name="serving-graph", kind="serving")

# Build the serving graph
graph = serving_fn.set_topology("flow")
graph.to(step).respond()